# unsupervised_contour_clustering_HDBSCAN.ipynb

After running this notebook you can:
- run view_tree_shape_classes.ipynb to see tree shape samples from each cluster
- have a look at some of the views set up in the database

## References
- [Gemini reference](https://share.gemini.google/OH0lJw2upRny)
- [Tuning DBSCAN parameters](https://share.gemini.google/vN3GhPCw2g3x)
- [Gemini reference for standard hdbscan module](https://share.google/aimode/EBb5xLhfQzgcLtEMP)

In [1]:
import cv2
import numpy as np
# from sklearn.cluster import HDBSCAN
# using the HDBSCAN implementation from the hdbscan package instead of sklearn because of bugs
import hdbscan
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.metrics import pairwise_distances_argmin_min
import matplotlib.pyplot as plt
import sqlite3
import json
from icecream import ic
import joblib
import shutil

In [2]:
db_path = '/home/aubrey/Desktop/Efate2025/Efate2025B.db'

In [3]:
def get_spatialite_contours(db_path, table_name, geom_column, additional_filters='', limit=100):
    """
    Fetches geometries from a SpatiaLite database and converts them into 
    a list of NumPy arrays structured for OpenCV contour functions.
    
    Parameters:
        db_path (str): Path to the SpatiaLite database file.
        table_name (str): Name of the table to query.
        geom_column (str): Name of the geometry column.
        additional_filters(str): optional filters to be added to WHERE; see example below
        limit (int): Maximum number of features to fetch (default 100).
        
    Returns:
        list of np.ndarray: A list of arrays, each with shape (N, 1, 2) and dtype int32.
        
    Example for additional_filters argument: 'AND confidence>0.5 AND tree_touches_edge=0'
    """
    contours = []
    tree_ids = []
    
    # 1. Connect and query the database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    conn.enable_load_extension(True)
    conn.load_extension('mod_spatialite')
    query = f"""
        SELECT AsGeoJSON({geom_column}), tree_id 
        FROM {table_name} 
        WHERE {geom_column} IS NOT NULL 
        {additional_filters}
        LIMIT {limit};
    """
    
    try:
        cursor.execute(query)
        rows = cursor.fetchall()
        
        # 2. Parse the results
        for row in rows:
            if not row[0]:
                continue
            
            tree_ids.append(row[1])    
            geojson_data = json.loads(row[0])
            geom_type = geojson_data.get("type")
            
            if geom_type == "Polygon":
                # Extract exterior ring coordinates
                coords = geojson_data["coordinates"][0]
                pts = np.array(coords, dtype=np.int32).reshape((-1, 1, 2))
                contours.append(pts)
                
            elif geom_type == "MultiPolygon":
                # Unroll each sub-polygon
                for polygon in geojson_data["coordinates"]:
                    coords = polygon[0]
                    pts = np.array(coords, dtype=np.int32).reshape((-1, 1, 2))
                    contours.append(pts)
                          
    finally:
        # Ensure the connection closes even if an error occurs
        conn.close()
        
    return contours, tree_ids

In [4]:
def extract_invariant_features(contour, log_transform=False):
    """
    Returns log-transformed Hu Moments from an OpenCV contour.
    These features are invariant to translation, scale, and rotation.
    Log transform is
    """
    moments = cv2.moments(contour)
    hu_moments = cv2.HuMoments(moments).flatten()
    if log_transform:
        log_hu = [-1.0 * np.sign(m) * np.log10(np.abs(m)) if m != 0 else 0.0 for m in hu_moments]
        return np.array(log_hu)
    else:
        return hu_moments

# Main

# Training the Model (Don't run this function unless you are sure you want to change the model.)

In [5]:
def train_model():
    """  
    Trains a HDBSCAN model to cluster tree shapes (polygons).
    Inputs are invariant Hu moments of polygons.
    Outputs are cluster indices.
    
    The trained model can be loaded from disk using code like this:
    loaded_pipeline = joblib.load('hdbscan_pipeline.joblib')
    scaler = loaded_pipeline.named_steps['scaler']
    hdbscan_model = loaded_pipeline.named_steps['hdbscan']
    """
    # Get tree contours and calculate invariant features (Hu moments)
    contours, tree_ids = get_spatialite_contours(
        db_path='/home/aubrey/Desktop/Efate2025/Efate2025B.db',
        table_name='trees',
        geom_column='tree_poly',
        additional_filters='AND confidence>0.4 AND tree_touches_edge=0 AND pixel_count > 400',
        limit=1000000   
    )
    features = np.array([extract_invariant_features(c) for c in contours])
    ic(features);

    # Create, run and save pipeline
    hdbscan_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('hdbscan', hdbscan.HDBSCAN(
            min_cluster_size=15, 
            prediction_data=True
        ))
    ])
    hdbscan_pipeline.fit(features)
    joblib.dump(hdbscan_pipeline, 'hdbscan_pipeline.joblib')
    
    # Usage example:
    # train_model()

## Using the trained HDBSCAN model to classify tree shapes

In [6]:
def classify_tree_shapes(db_path:str, model_path:str, cluster2class:dict):
    """  
    This function uses a trained HDBSCAN madel to assign a values in the trees.tree_poly filed into clusters.
    The shape is then assigned to a class using the cluster2class dictionary.
    """
    ic(db_path, model_path, cluster2class)
    
    conn = sqlite3.connect(db_path)
    conn.enable_load_extension(True)
    conn.load_extension('mod_spatialite')
    
    # get field names from trees table
    with conn:
        cursor = conn.execute("PRAGMA table_info(trees)")
        field_names = [row[1] for row in cursor.fetchall()]
        ic(field_names)
    
    # ensure fields exist
    with conn:
        if 'pixel_count' not in field_names:     conn.execute('ALTER TABLE trees ADD COLUMN pixel_count DOUBLE')
        if 'shape_class' not in field_names:     conn.execute('ALTER TABLE trees ADD COLUMN shape_class INTEGER')
        if 'soft_tree_class' not in field_names: conn.execute('ALTER TABLE trees ADD COLUMN soft_tree_class INTEGER')
        if 'soft_tree_prob' not in field_names:  conn.execute('ALTER TABLE trees ADD COLUMN soft_tree_prob REAL')
        if 'tree_class' not in field_names:      conn.execute('ALTER TABLE trees ADD COLUMN tree_class TEXT')
           
    # populate pixel_count field (= area of tree_poly in pixels)
    with conn:
        conn.execute('UPDATE trees SET pixel_count = ST_Area(tree_poly)')
    
    # Get tree contours and extract invariant features
    contours, tree_ids = get_spatialite_contours(
        db_path=db_path,
        table_name='trees',
        geom_column='tree_poly',
        additional_filters='AND confidence>0.4 AND tree_touches_edge=0 AND pixel_count > 400',
        limit=1000000   
    )
    ic(type(tree_ids), ic(len(tree_ids)));
    features = np.array([extract_invariant_features(c) for c in contours])
    ic(features);
    
    # 1. Load your originally saved pipeline (Scaler + HDBSCAN)
    loaded_pipeline = joblib.load(model_path)
    scaler = loaded_pipeline.named_steps['scaler']
    hdbscan_model = loaded_pipeline.named_steps['hdbscan']

    # 2. Incoming new raw data X (e.g., 3 new points)
    new_X = features
    new_X_scaled = scaler.transform(new_X)

    # 2. Generate a membership vector matrix for the new data
    # This returns a matrix of shape: (n_samples, n_clusters)
    soft_probabilities = hdbscan.membership_vector(hdbscan_model, new_X_scaled)
    ic(soft_probabilities);

    # 3. Clean up edge cases (Optional but highly recommended)
    # Outlier points or extreme noise may result in all zeros for a row
    # You can normalize them or identify them by checking the row sums
    row_sums = soft_probabilities.sum(axis=1)
    is_noise = row_sums == 0
    ic(f"Number of noise points: {np.sum(is_noise)}");
    
    # gets the index of largest value in each row: this will be the tree_class
    soft_tree_classes = np.argmax(soft_probabilities, axis=1) 
    ic(soft_tree_classes)
    
    # gets the largest value in each row: this will be the probability 
    soft_tree_probs = np.max(soft_probabilities, axis=1)
    ic(soft_tree_probs)
    
    ic(hdbscan_model.labels_)
    
    # Update database with predicted cluster labels and probabilities
    for tree_id in tree_ids:
        idx = tree_ids.index(tree_id)
        if idx % 1000 == 0:
            print(f"Processing tree_id: {tree_id} (index {idx})")
        shape_class = int(hdbscan_model.labels_[idx])
        soft_tree_class = int(soft_tree_classes[idx])
        soft_tree_prob = float(soft_tree_probs[idx])
        tree_class = cluster2class[soft_tree_class]
        cursor.execute(f""" 
            UPDATE trees 
            SET shape_class = {shape_class}, 
                soft_tree_class = {soft_tree_class}, 
                soft_tree_prob = {soft_tree_prob}, 
                tree_class = '{tree_class}'  
            WHERE tree_id = {tree_id}
        """)
    conn.commit()

    ic("Database updated with predicted cluster labels and probabilities.")    
    
    conn.close()
    
    return

    # Returns hard labels and a prediction strength vector (0.0 to 1.0)
    # labels, strengths = hdbscan.approximate_predict(hdbscan_model, new_X)
    # ic(labels, strengths);

    # Find the closest center index
    # closest_center_indices, distances = pairwise_distances_argmin_min(new_X_scaled, centers)
    # ic(closest_center_indices, distances);


In [7]:
# MAIN

# test on a copy of db_path
new_db_path = db_path.replace('.', '_copy.')
shutil.copy2(db_path, new_db_path)
db_path = new_db_path
ic(f'{db_path} copied to {new_db_path}');

# create cluster2class dict 
# figure out a better way of implementing this: in config or DB??
cluster2class = {}
for i in range(33):
    if i in [0,1,2,3,4,5,6,7,9,10,11,12,13,16,17,18,19,20,21]:
        cluster2class[i] = 'dead'
    else:
        cluster2class[i] = 'alive'

classify_tree_shapes(
    db_path=db_path, 
    model_path='hdbscan_pipeline.joblib', 
    cluster2class=cluster2class
)

# conn = sqlite3.connect(db_path)
# conn.enable_load_extension(True)
# conn.load_extension('mod_spatialite')
# cursor = conn.cursor()

ic| f'{db_path} copied to {new_db_path}': '/home/aubrey/Desktop/Efate2025/Efate2025B_copy.db copied to /home/aubrey/Desktop/Efate2025/Efate2025B_copy.db'
ic| db_path: '/home/aubrey/Desktop/Efate2025/Efate2025B_copy.db'
    model_path: 'hdbscan_pipeline.joblib'
    cluster2class: {0: 'dead',
                    1: 'dead',
                    2: 'dead',
                    3: 'dead',
                    4: 'dead',
                    5: 'dead',
                    6: 'dead',
                    7: 'dead',
                    8: 'alive',
                    9: 'dead',
                    10: 'dead',
                    11: 'dead',
                    12: 'dead',
                    13: 'dead',
                    14: 'alive',
                    15: 'alive',
                    16: 'dead',
                    17: 'dead',
                    18: 'dead',
                    19: 'dead',
                    20: 'dead',
                    21: 'dead',
                    22: 'alive',
         

Processing tree_id: 2 (index 0)
Processing tree_id: 2147 (index 1000)
Processing tree_id: 4580 (index 2000)
Processing tree_id: 6727 (index 3000)
Processing tree_id: 8948 (index 4000)
Processing tree_id: 11190 (index 5000)
Processing tree_id: 13554 (index 6000)
Processing tree_id: 15812 (index 7000)
Processing tree_id: 18012 (index 8000)
Processing tree_id: 20275 (index 9000)
Processing tree_id: 22704 (index 10000)
Processing tree_id: 25178 (index 11000)


ic| "Database updated with predicted cluster labels and probabilities.": 'Database updated with predicted cluster labels and probabilities.'


In [8]:
ic(soft_tree_classes)
ic(soft_tree_probs);

NameError: name 'soft_tree_classes' is not defined

# Extra stuff below

In [ ]:
# # create a new column in the trees table and populate with area of tree_poly in pixels
# cursor.execute('ALTER TABLE trees ADD COLUMN pixel_count DOUBLE;')
# conn.commit()
# cursor.execute('UPDATE trees SET pixel_count = ST_Area(tree_poly);')
# conn.commit()

In [ ]:
# # Use the trained pipeline to predict clusters for new data points
# # In this case, we use the same features for demonstration, but in practice, this would be new incoming data.

# # 1. Load your originally saved pipeline (Scaler + HDBSCAN)
# loaded_pipeline = joblib.load('hdbscan_pipeline.joblib')

# # Extract components from the pipeline
# scaler = loaded_pipeline.named_steps['scaler']
# hdbscan_model = loaded_pipeline.named_steps['hdbscan']

# # 2. Incoming new raw data X (e.g., 3 new points)
# new_X = features
# new_X_scaled = scaler.transform(new_X)

# # 2. Generate a membership vector matrix for the new data
# # This returns a matrix of shape: (n_samples, n_clusters)
# soft_probabilities = hdbscan.membership_vector(hdbscan_model, new_X_scaled)
# ic(soft_probabilities);

# # 3. Clean up edge cases (Optional but highly recommended)
# # Outlier points or extreme noise may result in all zeros for a row
# # You can normalize them or identify them by checking the row sums
# row_sums = soft_probabilities.sum(axis=1)
# is_noise = row_sums == 0
# ic(f"Number of noise points: {np.sum(is_noise)}");
# soft_tree_classes = np.argmax(soft_probabilities, axis=1)
# soft_tree_probs = np.max(soft_probabilities, axis=1)
# ic(soft_tree_classes, soft_tree_probs);



# # Returns hard labels and a prediction strength vector (0.0 to 1.0)
# # labels, strengths = hdbscan.approximate_predict(hdbscan_model, new_X)
# # ic(labels, strengths);

# # Find the closest center index
# # closest_center_indices, distances = pairwise_distances_argmin_min(new_X_scaled, centers)
# # ic(closest_center_indices, distances);

In [ ]:
new_X_scaled = scaler.transform(new_X)

In [ ]:
ic(hdbscan_model.labels_[0])
ic(soft_tree_classes[0]);
ic(soft_probabilities[0]);
ic(np.max(soft_probabilities[0]));

In [ ]:
ic(hdbscan_model.labels_, hdbscan_model.probabilities_, hdbscan_model.outlier_scores_);

In [ ]:
def soft_tree_class_to_tree_class(soft_tree_class):
    """
    Maps soft_tree_class to a binary tree_class label.
    
    Args:
        soft_tree_class (int): The soft tree class label.
        
    Returns:
        str: 'dead' if the soft_tree_class is in the specified list, else 'alive'.
    """
    if soft_tree_class in [0,1,2,3,4,5,6,7,9,10,11,12,13,16,17,18,19,20,21]:
        return 'dead'
    else:
        return 'alive'
    
# Add new columns to trees table
with conn:
    conn.execute('ALTER TABLE trees ADD COLUMN shape_class INTEGER')
    conn.execute('ALTER TABLE trees ADD COLUMN soft_tree_class INTEGER')
    conn.execute('ALTER TABLE trees ADD COLUMN soft_tree_prob REAL')
    conn.execute('ALTER TABLE trees ADD COLUMN tree_class TEXT')

# Update database with predicted cluster labels and distances
for tree_id in tree_ids:
    idx = tree_ids.index(tree_id)
    if idx % 1000 == 0:
        print(f"Processing tree_id: {tree_id} (index {idx})")
    shape_class = int(hdbscan_model.labels_[idx])
    soft_tree_class = int(soft_tree_classes[idx])
    soft_tree_prob = float(soft_tree_probs[idx])
    tree_class = soft_tree_class_to_tree_class(soft_tree_class)
    cursor.execute(f""" 
        UPDATE trees 
        SET shape_class = {shape_class}, 
            soft_tree_class = {soft_tree_class}, 
            soft_tree_prob = {soft_tree_prob}, 
            tree_class = '{tree_class}'  
        WHERE tree_id = {tree_id}
    """)
conn.commit()

print("Database updated with predicted cluster labels and distances.")